In [1]:
from frequent_itemsets import get_frequent_itemsets
from association_rules import get_association_rules
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment


frequent_itemsets = get_frequent_itemsets(min_sup=0.6)
association_rules = get_association_rules(frequent_itemsets, min_confidence=0.9)

frequent_itemsets['itemsets'] = frequent_itemsets['itemsets'].apply(lambda x: ', '.join(sorted(x)))
frequent_itemsets['support'] = frequent_itemsets['support'].round(4)

association_rules = association_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
association_rules['antecedents'] = association_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
association_rules['consequents'] = association_rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

for col in ['support', 'confidence', 'lift']:
    if col in association_rules.columns:
        association_rules[col] = association_rules[col].round(4)


output_path = "./Output/subject_taken_together.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    frequent_itemsets.to_excel(writer, sheet_name='Frequent Itemsets', index=False)
    association_rules.to_excel(writer, sheet_name='Association Rules', index=False)


wb = load_workbook(output_path)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')

    for col in ws.columns:
        max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
        ws.column_dimensions[col[0].column_letter].width = max_len + 2

wb.save(output_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from course_frequency import get_course_frequencies
import pandas as pd
import os
import xlsxwriter
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment

output_dir = "./Output"
os.makedirs(output_dir, exist_ok=True)
excel_path = os.path.join(output_dir, "course_frequency.xlsx")

overall_counts, by_major_counts, by_semester_counts = get_course_frequencies()


# Chart 1: Overall Frequency
plt.figure(figsize=(21, 9))
overall_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Frequency of All Courses Taken by Students Across All Semesters', fontsize=16)
plt.xlabel('Course Code', fontsize=14)
plt.ylabel('Number of Students', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
overall_img = os.path.join(output_dir, "overall_counts.png")
plt.savefig(overall_img, dpi=300)
plt.close()

# Chart 2: By Major
courses = by_major_counts.index
x = np.arange(len(courses))
width = 0.35
fig, ax = plt.subplots(figsize=(21, 9))
ax.bar(x - width/2, by_major_counts['Primary'], width, label='Primary', color='skyblue', edgecolor='black')
ax.bar(x + width/2, by_major_counts['NonPrimary'], width, label='NonPrimary', color='lightcoral', edgecolor='black')
ax.set_title('Course Enrollment by Major Type', fontsize=16)
ax.set_xlabel('Course Code', fontsize=14)
ax.set_ylabel('Number of Students', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(courses, rotation=45)
ax.set_xlim(-0.5, len(courses) - 0.5)
ax.legend()
plt.tight_layout()
major_img = os.path.join(output_dir, "by_major_counts.png")
plt.savefig(major_img, dpi=300)
plt.close()

# Chart 3: Per Semester
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(21, 21))
axes = axes.flatten()
for i, (semester, counts) in enumerate(by_semester_counts.items()):
    ax = axes[i]
    counts.plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
    ax.set_title(f'Frequency of Course in {semester}', fontsize=14)
    ax.set_xlabel('Course Code')
    ax.set_ylabel('Number of Students')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
semester_img = os.path.join(output_dir, "by_semester_counts.png")
plt.savefig(semester_img, dpi=300)
plt.close()

# ========== EXPORT TO EXCEL ==========
workbook = xlsxwriter.Workbook(excel_path)

# Sheet: Overall Frequency
ws1 = workbook.add_worksheet("Overall Frequency")
ws1.write_column("A1", overall_counts.index)
ws1.write_column("B1", overall_counts.values)
ws1.write("A1", "Course Code")
ws1.write("B1", "Student Count")

# Sheet: By Major
ws2 = workbook.add_worksheet("By Major")
ws2.write_row("A1", ["Course Code", "Primary", "NonPrimary"])
for i, (course, row) in enumerate(by_major_counts.iterrows(), start=1):
    ws2.write(i, 0, course)
    ws2.write(i, 1, row["Primary"])
    ws2.write(i, 2, row["NonPrimary"])

# Sheet: By Semester
ws3 = workbook.add_worksheet("By Semester")
start_col = 0
for semester, counts in by_semester_counts.items():
    ws3.write(0, start_col, f"{semester} - Course")
    ws3.write(0, start_col + 1, f"{semester} - Count")
    for i, (course, count) in enumerate(counts.items(), start=1):
        ws3.write(i, start_col, course)
        ws3.write(i, start_col + 1, count)
    start_col += 3

# Sheet: Charts
ws4 = workbook.add_worksheet("Charts")
ws4.insert_image("A1", overall_img, {'x_scale': 0.7, 'y_scale': 0.7})
ws4.insert_image("A40", major_img, {'x_scale': 0.7, 'y_scale': 0.7})
ws4.insert_image("A80", semester_img, {'x_scale': 0.6, 'y_scale': 0.6})

workbook.close()

wb = load_workbook(excel_path)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    
    for col in ws.columns:
        max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
        ws.column_dimensions[col[0].column_letter].width = max_len + 2

wb.save(excel_path)


FileNotFoundError: [Errno 2] No such file or directory: './Output\\overall_counts.png'